# 📘 Deep Learning Text Generation Learning Project
## Text Generation using **Vanilla RNN, LSTM, and GRU**

This notebook is built for **students and beginners** to understand how sequence models learn:
- grammar
- sentence flow
- contextual dependencies
- next-word prediction
- text generation

🎯 **Goal:** Compare **Simple RNN vs LSTM vs GRU** on the same text corpus and understand why gated architectures perform better.

# 🧠 Problem Statement
Design and implement a DL model capable of learning the underlying structure, grammar, and contextual dependencies of a given text corpus to generate coherent and meaningful text sequences using:

1. **Vanilla RNN**
2. **LSTM**
3. **GRU**

Then compare:
- training loss
- generated text quality
- memory handling
- long-term dependency learning

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import numpy as np
import matplotlib.pyplot as plt
print("TensorFlow:", tf.__version__)

# 📥 Load Text Corpus
We use a **small built-in sample corpus** so students can run this quickly.
You can later replace it with:
- Shakespeare text
- song lyrics
- chatbot data
- story paragraphs
- custom PDF extracted text

In [ ]:
corpus = '''
deep learning is transforming artificial intelligence
recurrent neural networks are useful for sequential data
lstm helps remember long term dependencies
gru is faster and simpler than lstm
text generation models predict the next word
deep learning models can generate meaningful sentences
'''
print(corpus)

# 🔤 Tokenization & Sequence Creation
We convert text into integer tokens and create **n-gram style sequences**
for next-word prediction.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

input_sequences = []
for line in corpus.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

# 🧠 Model 1: Vanilla RNN
This is the baseline sequential model.
It struggles with long-term dependencies because of vanishing gradients.

In [ ]:
rnn_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    SimpleRNN(64),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

rnn_history = rnn_model.fit(X, y, epochs=100, verbose=0)
print("Vanilla RNN training completed")

# 🔒 Model 2: LSTM
LSTM uses **input, forget, and output gates**
to preserve long-term memory.

In [ ]:
lstm_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    LSTM(64),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

lstm_history = lstm_model.fit(X, y, epochs=100, verbose=0)
print("LSTM training completed")

# ⚡ Model 3: GRU
GRU uses **reset + update gates**.
It is computationally faster than LSTM and often gives similar results.

In [ ]:
gru_model = Sequential([
    Embedding(total_words, 32, input_length=max_len-1),
    GRU(64),
    Dense(total_words, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

gru_history = gru_model.fit(X, y, epochs=100, verbose=0)
print("GRU training completed")

## 📉 Compare Training Loss

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(rnn_history.history['loss'], label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'], label='GRU')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

# ✍️ Text Generation Function
This function predicts the next word repeatedly to generate a sentence.

In [ ]:
def generate_text(model, seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

## 🧪 Generate Text Samples

In [ ]:
print("RNN :", generate_text(rnn_model, "deep learning", 5))
print("LSTM:", generate_text(lstm_model, "deep learning", 5))
print("GRU :", generate_text(gru_model, "deep learning", 5))

## 🧪 Additional: PyTorch character-level models (RNN / LSTM / GRU)
The following cells implement a compact character-level `CharRNN` in PyTorch, a minimal training loop, and a sampling function. This section complements the word-level TensorFlow examples above and is useful to explore sequence modelling at the character level.

In [ ]:
# Check PyTorch availability
try:
    import torch
    import torch.nn as nn
    print('PyTorch version:', torch.__version__)
except Exception as e:
    print('PyTorch import failed:', e)
    print('If missing, install with: pip install torch')

In [ ]:
# CharRNN implementation (vanilla RNN / LSTM / GRU)
import torch
import torch.nn as nn

class CharRNN(nn.Module):
    def __init__(self, n_vocab, embed_size=64, hidden_size=128, n_layers=2, rnn_type='LSTM', dropout=0.1):
        super().__init__()
        self.n_vocab = n_vocab
        self.embed = nn.Embedding(n_vocab, embed_size)
        rnn_type = rnn_type.upper()
        self.rnn_type = rnn_type
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_size, hidden_size, n_layers, batch_first=True, dropout=dropout)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(embed_size, hidden_size, n_layers, batch_first=True, dropout=dropout)
        else:
            self.rnn = nn.RNN(embed_size, hidden_size, n_layers, nonlinearity='tanh', batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, n_vocab)

    def forward(self, x, hidden=None):
        emb = self.embed(x)
        out, hidden = self.rnn(emb, hidden)
        out = out.contiguous().view(out.size(0) * out.size(1), out.size(2))
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size, device):
        num_layers = self.rnn.num_layers
        hidden_size = self.rnn.hidden_size
        if self.rnn_type == 'LSTM':
            return (torch.zeros(num_layers, batch_size, hidden_size, device=device),
                    torch.zeros(num_layers, batch_size, hidden_size, device=device))
        else:
            return torch.zeros(num_layers, batch_size, hidden_size, device=device)

In [ ]:
# Prepare character-level dataset from existing `corpus` variable
from collections import Counter
import torch
from torch.utils.data import DataLoader, TensorDataset

text = corpus.strip() if 'corpus' in globals() else 'hello world this is a sample corpus for char rnn'
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
seq_len = 40
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
inputs = []
targets = []
for i in range(0, len(data) - seq_len):
    inputs.append(data[i:i+seq_len])
    targets.append(data[i+1:i+seq_len+1])
if len(inputs) == 0:
    # fallback: shorter seq_len
    seq_len = max(2, len(data) - 1)
    inputs = [data[:seq_len]]
    targets = [data[1:seq_len+1]]
X = torch.stack(inputs)
Y = torch.stack(targets)
batch_size = 16
dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print('Vocab size:', len(stoi), 'seq_len:', seq_len, 'samples:', len(X))

In [ ]:
# Small training loop (quick smoke run). Adjust epochs for real training.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CharRNN(len(stoi), embed_size=64, hidden_size=128, n_layers=2, rnn_type='LSTM')
model.to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs = 6
for epoch in range(1, epochs+1):
    model.train()
    total = 0.0
    for bx, by in dataloader:
        bx = bx.to(device)
        by = by.to(device)
        batch_sz = bx.size(0)
        hidden = model.init_hidden(batch_sz, device)
        optim.zero_grad()
        logits, _ = model(bx, hidden)
        loss = criterion(logits, by.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optim.step()
        total += loss.item()
    print(f'Epoch {epoch}/{epochs} — loss: {total/len(dataloader):.4f}')

In [ ]:
# Sampling / generation helper
import torch.nn.functional as F

def generate_text(model, seed, length=200, temperature=1.0, device=None):
    device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
    model.eval()
    hidden = model.init_hidden(1, device)
    # prime with seed
    if len(seed) == 0:
        seed = list(itos.values())[:1][0]
    input_seq = torch.tensor([[stoi.get(ch, 0) for ch in seed]], dtype=torch.long, device=device)
    generated = seed
    with torch.no_grad():
        for i in range(input_seq.size(1)):
            _, hidden = model(input_seq[:, i:i+1], hidden)
        last = input_seq[:, -1]
        for _ in range(length):
            out, hidden = model(last.unsqueeze(1), hidden)
            logits = out.squeeze(0) / max(1e-8, temperature)
            probs = F.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, 1).item()
            ch = itos[idx]
            generated += ch
            last = torch.tensor([idx], dtype=torch.long, device=device)
    return generated

In [ ]:
# Generate a short sample from the trained model
seed = 'deep ' if 'corpus' in globals() else 'h'
print(generate_text(model, seed, length=300, temperature=0.9, device=device))

# 📚 Student Learning Tasks
### ✅ Beginner Tasks
1. Replace corpus with your own paragraph
2. Increase embedding dimension
3. Increase epochs to 200
4. Change hidden units 64 → 128
5. Generate 10 words instead of 5

# ✅ Conclusion
- **Vanilla RNN** learns short patterns but struggles with memory
- **LSTM** captures long-range grammar dependencies better
- **GRU** gives similar performance with fewer gates and faster training
- This notebook helps students understand **sequence modeling mathematically and practically**